# Ridge Regression

In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
import os

In [4]:
df_train = pd.read_csv('household_train.csv', parse_dates=['Datetime'])
df_valid = pd.read_csv('household_validation.csv', parse_dates=['Datetime'])
df_test = pd.read_csv('household_test.csv', parse_dates=['Datetime'])

### Korrelation der wichtigsten Features

In [6]:
# 1. Nur numerische Spalten auswählen
numeric_columns = df_train.select_dtypes(include=[np.number]).columns.tolist()

print("Numerische Spalten:")
print(numeric_columns)

# 2. Korrelationsanalyse nur mit numerischen Spalten
correlation_with_target = df_train[numeric_columns].corr()[Global_active_power].abs().sort_values(ascending=False)

print("\nKorrelation mit Global_active_power:")
print(correlation_with_target)

# 3. Visualisierung
plt.figure(figsize=(10, 6))
correlation_with_target.drop(Global_active_power).plot(kind='bar')
plt.title("Korrelation mit Global_active_power")
plt.ylabel("Absolute Korrelation")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Numerische Spalten:
['Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3', 'Wochentag', 'Wochenende', 'Jahr', 'Monat']


NameError: name 'Global_active_power' is not defined

### 30 minütige Vorhersage

In [ ]:
df_train.set_index('Datetime', inplace=True)
df_valid.set_index('Datetime', inplace=True)
df_test.set_index('Datetime', inplace=True)

print(f"Training: {len(df_train)} Zeilen")
print(f"Validation: {len(df_valid)} Zeilen")
print(f"Test: {len(df_test)} Zeilen")

# --- 3. LAG-FEATURES ERSTELLEN (Multivariat!) ---
features_for_lags = [
    "Global_intensity",      # 🏆 Wichtigstes Feature!
    "Sub_metering_3",        # 🥈 Sehr wichtig
    "Sub_metering_1",        # 🥉 Wichtig
    "Sub_metering_2",        # 🥉 Wichtig
    "Global_active_power"    # Zielvariable für Lags
]

# Diese Features KÖNNEN WEGGELASSEN werden:
# "Voltage", "Global_reactive_power", "Wochentag", "Jahr", "Monat"

n_lags = 30  # 30 Minuten Lags
target = "Global_active_power"

def create_lag_features(df, features, n_lags):
    df_lagged = df.copy()
    for feature in features:
        for lag in range(1, n_lags + 1):
            df_lagged[f'{feature}_lag_{lag}'] = df_lagged[feature].shift(lag)
    return df_lagged

print("Erstelle Lag-Features...")
df_train_lagged = create_lag_features(df_train, features_for_lags, n_lags)
df_valid_lagged = create_lag_features(df_valid, features_for_lags, n_lags)
df_test_lagged = create_lag_features(df_test, features_for_lags, n_lags)

# Entferne NaN durch Shifting
df_train_clean = df_train_lagged.dropna()
df_valid_clean = df_valid_lagged.dropna()
df_test_clean = df_test_lagged.dropna()

print(f"Training nach Lagging: {len(df_train_clean)} Zeilen")

# --- 4. FEATURES VORBEREITEN ---
# Wähle alle Lag-Features
lag_columns = [col for col in df_train_clean.columns if 'lag_' in col]

# Zusätzliche zeitliche Features (falls vorhanden)
time_features = ['hour', 'day_of_week', 'Wochenende']
available_time_features = [f for f in time_features if f in df_train_clean.columns]

# Kombiniere alle Features
all_features = lag_columns + available_time_features

print(f"\nAnzahl Features: {len(all_features)}")

# Definiere X und y
X_train = df_train_clean[all_features]
y_train = df_train_clean[target]

X_val = df_valid_clean[all_features]
y_val = df_valid_clean[target]

X_test = df_test_clean[all_features]
y_test = df_test_clean[target]

# --- 5. SKALIERUNG mit float32 ---
print("Skaliere Features (float32)...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.astype(np.float32))  # float32 statt float64
X_val_scaled = scaler.transform(X_val.astype(np.float32))
X_test_scaled = scaler.transform(X_test.astype(np.float32))
# --- 5. SKALIERUNG ---

In [ ]:
# --- 6. MULTI-STEP FORECASTING (30 Minuten) ---
forecast_horizon = 30

def create_multi_step_y(y, horizon):
    n_samples = len(y) - horizon + 1
    y_multi = np.zeros((n_samples, horizon))
    for i in range(n_samples):
        y_multi[i] = y.iloc[i:i+horizon]
    return y_multi

y_train_multi = create_multi_step_y(y_train, forecast_horizon)
y_val_multi = create_multi_step_y(y_val, forecast_horizon)
y_test_multi = create_multi_step_y(y_test, forecast_horizon)

# Passe Feature-Matrizen an
X_train_multi = X_train_scaled[:-(forecast_horizon - 1) if (forecast_horizon - 1) > 0 else None]
X_val_multi = X_val_scaled[:-(forecast_horizon - 1) if (forecast_horizon - 1) > 0 else None]
X_test_multi = X_test_scaled[:-(forecast_horizon - 1) if (forecast_horizon - 1) > 0 else None]

# --- 7. RIDGE REGRESSION TRAINING ---
print("\nTrainiere Ridge Regression...")
ridge_model = RidgeCV(
    alphas=[0.01, 0.1, 1.0, 10.0, 100.0, 1000.0],
    cv=5
)

ridge_model.fit(X_train_multi, y_train_multi)
print(f"Beste Alpha: {ridge_model.alpha_}")

# --- 8. VORHERSAGEN & EVALUATION ---
y_val_pred = ridge_model.predict(X_val_multi)
y_test_pred = ridge_model.predict(X_test_multi)

# Metriken berechnen
def calculate_metrics(y_true, y_pred, set_name):
    mae_per_step = mean_absolute_error(y_true, y_pred, multioutput='raw_values')
    mae_total = mean_absolute_error(y_true, y_pred)
    
    print(f"\n{set_name} MAE pro Minute (1-{forecast_horizon}):")
    for i, mae in enumerate(mae_per_step, 1):
        print(f"  Minute {i}: {mae:.4f} kW")
    
    print(f"{set_name} Gesamt-MAE: {mae_total:.4f} kW")
    return mae_per_step, mae_total

mae_val_per_step, mae_val_total = calculate_metrics(y_val_multi, y_val_pred, "VALIDATION")
mae_test_per_step, mae_test_total = calculate_metrics(y_test_multi, y_test_pred, "TEST")

# --- 9. VISUALISIERUNG ---
plt.figure(figsize=(12, 6))
plt.plot(range(1, forecast_horizon + 1), mae_val_per_step, 'o-', label='Validation MAE', linewidth=2)
plt.plot(range(1, forecast_horizon + 1), mae_test_per_step, 'o-', label='Test MAE', linewidth=2)
plt.xlabel('Vorhersage-Schritt (Minute in die Zukunft)')
plt.ylabel('Mean Absolute Error (kW)')
plt.title(f'Ridge Regression Performance: {forecast_horizon}-Minuten Vorhersage\nMultivariat mit {n_lags} Lags')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(range(1, forecast_horizon + 1, 5))
plt.tight_layout()
plt.savefig('ridge_regression_performance.png')
plt.show()

# --- 10. FEATURE IMPORTANCE ---
feature_importance = pd.DataFrame({
    'feature': all_features,
    'coefficient': ridge_model.coef_[0]  # Gewichte für ersten Vorhersageschritt
})
feature_importance['abs_coefficient'] = np.abs(feature_importance['coefficient'])
feature_importance = feature_importance.sort_values('abs_coefficient', ascending=False)

print("\nTop 10 wichtigste Features:")
print(feature_importance.head(10)[['feature', 'coefficient']].to_string(index=False))

# --- 11. ERGEBNISSE SPEICHERN ---
results = {
    'model': 'Ridge Regression',
    'best_alpha': ridge_model.alpha_,
    'validation_mae': mae_val_total,
    'test_mae': mae_test_total,
    'n_features': len(all_features),
    'n_lags': n_lags
}

print(f"\n✅ Training abgeschlossen!")
print(f"Beste Alpha: {results['best_alpha']}")
print(f"Test MAE: {results['test_mae']:.4f} kW")
# Vorhersagen vs. Realität plotten
plt.figure(figsize=(12, 6))
plt.plot(y_test.values[:100], label='Echte Werte')
plt.plot(y_test_pred[:100], label='Vorhersagen')
plt.title("Vorhersage vs. Realität (erste 100 Werte)")
plt.legend()
plt.show()
from sklearn.metrics import mean_squared_error

rmse_test = np.sqrt(mean_squared_error(y_test_multi, y_test_pred))
print(f"Test RMSE: {rmse_test:.4f} kW")

In [ ]:
# RMSE pro Schritt
rmse_per_step = np.sqrt(mean_squared_error(y_test_multi, y_test_pred, multioutput='raw_values'))
def mean_absolute_percentage_error(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1e-8))) * 100  # Avoid division by zero

mape_test = mean_absolute_percentage_error(y_test_multi, y_test_pred)
print(f"Test MAPE: {mape_test:.2f}%")



In [ ]:
# MAPE pro Schritt

mape_per_step = np.array([mean_absolute_percentage_error(y_test_multi[:, i], y_test_pred[:, i]) for i in range(forecast_horizon)])
from sklearn.metrics import r2_score

r2_test = r2_score(y_test_multi, y_test_pred)
print(f"Test R²: {r2_test:.4f}")

In [ ]:

# R² pro Schritt  
r2_per_step = np.array([r2_score(y_test_multi[:, i], y_test_pred[:, i]) for i in range(forecast_horizon)])
import pandas as pd


In [ ]:
metrics = {
    'MAE': mae_test_total,
    'RMSE': rmse_test,
    'MAPE (%)': mape_test,
    'R²': r2_test
}

metrics_df = pd.DataFrame.from_dict(metrics, orient='index', columns=['Wert'])
metrics_df.index.name = 'Metrik'
print(metrics_df.round(4))

In [ ]:
# MAPE nur für Werte > 1 kW berechnen
mask = y_test_multi > 1.0  # Nur Verbrauch > 1 kW
mape_realistic = mean_absolute_percentage_error(y_test_multi[mask], y_test_pred[mask])
print(f"MAPE (>1 kW): {mape_realistic:.2f}%")

In [ ]:

# Histogramm der Fehler
errors = y_test_multi - y_test_pred
plt.hist(errors.flatten(), bins=50)
plt.title("Verteilung der Vorhersagefehler")
plt.xlabel("Fehler (kW)")
plt.ylabel("Häufigkeit")
plt.show()

In [ ]:
# Korrektur: errors ist 2D, wir müssen den Index umrechnen
max_error_flat_idx = np.argmax(np.abs(errors))  # Index im geflatetten Array
max_error_idx = np.unravel_index(max_error_flat_idx, errors.shape)  # Zurück zu 2D-Index

# Zeitpunkt berechnen (unter Berücksichtigung des Forecast Horizonts)
time_index = max_error_idx[0]  # Zeilenindex
forecast_step = max_error_idx[1]  # Schritt im Forecast

# Der tatsächliche Zeitpunkt
actual_time = df_test_clean.index[time_index + forecast_step]  # WICHTIG: + forecast_step!

print(f"Größter Fehler: {np.abs(errors).max():.2f} kW")
print(f"Zeitpunkt: {actual_time}")
print(f"Vorhergesagter Wert: {y_test_pred[time_index, forecast_step]:.2f} kW")
print(f"Tatsächlicher Wert: {y_test_multi[time_index, forecast_step]:.2f} kW")
# Plot um den Zeitpunkt des größten Fehlers
start_idx = max(0, time_index - 60)  # 60 Minuten vorher
end_idx = min(len(df_test_clean), time_index + 60)  # 60 Minuten nachher

plt.figure(figsize=(12, 6))
plt.plot(df_test_clean.index[start_idx:end_idx], df_test_clean[target][start_idx:end_idx], label='Tatsächlicher Verbrauch')
plt.axvline(x=actual_time, color='red', linestyle='--', label='Zeitpunkt maximaler Fehler')
plt.title("Verbrauch um den Zeitpunkt des größten Vorhersagefehlers")
plt.xlabel("Zeit")
plt.ylabel("Verbrauch (kW)")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:

# Plot des Extrem-Events
event_time = pd.to_datetime('2010-11-20 18:22:00')
start_time = event_time - pd.Timedelta(minutes=30)
end_time = event_time + pd.Timedelta(minutes=30)

# Daten für diesen Zeitraum
mask = (df_test_clean.index >= start_time) & (df_test_clean.index <= end_time)
event_data = df_test_clean.loc[mask]

plt.figure(figsize=(14, 8))
plt.plot(event_data.index, event_data[target], 'o-', linewidth=2, label='Tatsächlicher Verbrauch')
plt.axvline(x=event_time, color='red', linestyle='--', linewidth=2, label='Extrem-Spitze (9.72 kW)')
plt.title("Extrem-Spitze im Energieverbrauch: Samstag Abend 18:22 Uhr\n(Kochzeit mit mehreren Großgeräten gleichzeitig)", fontsize=14)
plt.xlabel("Zeit")
plt.ylabel("Verbrauch (kW)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('extrem_spitze_analysis.png', dpi=300, bbox_inches='tight')
plt.show()


### 24h Vorhersage

In [ ]:
# --- KONFIGURATION FÜR 24-STUNDEN-VORHERSAGE ---

# 1. Nur numerische Spalten für das Resampling auswählen
numeric_features = [
    "Global_active_power", "Global_reactive_power", "Voltage",
    "Global_intensity", "Sub_metering_1", "Sub_metering_2", "Sub_metering_3"
]

print("Resample auf stündliche Daten (nur numerische Spalten)...")
df_train_hourly = df_train[numeric_features].resample('H').mean()
df_valid_hourly = df_valid[numeric_features].resample('H').mean()
df_test_hourly = df_test[numeric_features].resample('H').mean()

# 2. Zeitliche Features später neu hinzufügen
df_train_hourly['hour'] = df_train_hourly.index.hour
df_train_hourly['day_of_week'] = df_train_hourly.index.dayofweek
df_train_hourly['is_weekend'] = df_train_hourly['day_of_week'].isin([5, 6]).astype(int)

df_valid_hourly['hour'] = df_valid_hourly.index.hour
df_valid_hourly['day_of_week'] = df_valid_hourly.index.dayofweek
df_valid_hourly['is_weekend'] = df_valid_hourly['day_of_week'].isin([5, 6]).astype(int)

df_test_hourly['hour'] = df_test_hourly.index.hour
df_test_hourly['day_of_week'] = df_test_hourly.index.dayofweek
df_test_hourly['is_weekend'] = df_test_hourly['day_of_week'].isin([5, 6]).astype(int)

# 3. Parameter für 24h-Vorhersage
forecast_horizon = 24  # 24 Stunden Vorhersage
n_lags = 48            # 48 Stunden History (2 Tage)
target = "Global_active_power"

# 4. Features für stündliche Vorhersage
features_for_lags = [
    "Global_active_power", 
    "Global_intensity",
    "Sub_metering_1", 
    "Sub_metering_2",
    "Sub_metering_3"
]

# 5. Lag-Features erstellen
print("Erstelle Lag-Features für stündliche Daten...")
def create_lag_features(df, features, n_lags):
    df_lagged = df.copy()
    for feature in features:
        for lag in range(1, n_lags + 1):
            df_lagged[f'{feature}_lag_{lag}'] = df_lagged[feature].shift(lag)
    return df_lagged

df_train_lagged = create_lag_features(df_train_hourly, features_for_lags, n_lags)
df_valid_lagged = create_lag_features(df_valid_hourly, features_for_lags, n_lags)
df_test_lagged = create_lag_features(df_test_hourly, features_for_lags, n_lags)

# 6. Daten cleaning
df_train_clean = df_train_lagged.dropna()
df_valid_clean = df_valid_lagged.dropna()
df_test_clean = df_test_lagged.dropna()

print(f"Stündliche Daten - Training: {len(df_train_clean)} Zeilen")
print(f"Stündliche Daten - Test: {len(df_test_clean)} Zeilen")
# --- 7. FEATURES VORBEREITEN FÜR 24H ---

# Wähle alle Lag-Features
lag_columns = [col for col in df_train_clean.columns if 'lag_' in col]

# Wähle zeitliche Features
time_features = ['hour', 'day_of_week', 'is_weekend']

# Kombiniere alle Features
all_features = lag_columns + time_features

print(f"Anzahl Features für 24h-Vorhersage: {len(all_features)}")

# Definiere X und y
X_train = df_train_clean[all_features]
y_train = df_train_clean[target]

X_val = df_valid_clean[all_features]
y_val = df_valid_clean[target]

X_test = df_test_clean[all_features]
y_test = df_test_clean[target]

# --- 8. SKALIERUNG ---
print("Skaliere Features...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# --- 9. MULTI-STEP FORECASTING (24 Stunden) ---
forecast_horizon = 24

def create_multi_step_y(y, horizon):
    n_samples = len(y) - horizon + 1
    y_multi = np.zeros((n_samples, horizon))
    for i in range(n_samples):
        y_multi[i] = y.iloc[i:i+horizon]
    return y_multi

y_train_multi = create_multi_step_y(y_train, forecast_horizon)
y_val_multi = create_multi_step_y(y_val, forecast_horizon)
y_test_multi = create_multi_step_y(y_test, forecast_horizon)

# Passe Feature-Matrizen an
X_train_multi = X_train_scaled[:-(forecast_horizon - 1) if (forecast_horizon - 1) > 0 else None]
X_val_multi = X_val_scaled[:-(forecast_horizon - 1) if (forecast_horizon - 1) > 0 else None]
X_test_multi = X_test_scaled[:-(forecast_horizon - 1) if (forecast_horizon - 1) > 0 else None]

# --- 10. RIDGE REGRESSION TRAINING FÜR 24H ---
print("\nTrainiere Ridge Regression für 24h-Vorhersage...")
ridge_model_24h = RidgeCV(
    alphas=[0.01, 0.1, 1.0, 10.0, 100.0, 1000.0],
    cv=5
)

ridge_model_24h.fit(X_train_multi, y_train_multi)
print(f"Beste Alpha: {ridge_model_24h.alpha_}")

# --- 11. VORHERSAGEN & EVALUATION ---
y_val_pred_24h = ridge_model_24h.predict(X_val_multi)
y_test_pred_24h = ridge_model_24h.predict(X_test_multi)

# Metriken berechnen
mae_val_per_step_24h, mae_val_total_24h = calculate_metrics(y_val_multi, y_val_pred_24h, "VALIDATION 24H")
mae_test_per_step_24h, mae_test_total_24h = calculate_metrics(y_test_multi, y_test_pred_24h, "TEST 24H")

# --- 12. VISUALISIERUNG 24H ---
plt.figure(figsize=(12, 6))
plt.plot(range(1, forecast_horizon + 1), mae_val_per_step_24h, 'o-', label='Validation MAE', linewidth=2)
plt.plot(range(1, forecast_horizon + 1), mae_test_per_step_24h, 'o-', label='Test MAE', linewidth=2)
plt.xlabel('Vorhersage-Schritt (Stunde in die Zukunft)')
plt.ylabel('Mean Absolute Error (kW)')
plt.title(f'Ridge Regression Performance: 24-Stunden Vorhersage\nMultivariat mit {n_lags} Lags')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(range(1, forecast_horizon + 1, 4))
plt.tight_layout()
plt.savefig('ridge_regression_24h_performance.png', dpi=300, bbox_inches='tight')
plt.show()

# --- 13. VERGLEICH 30MIN vs 24H ---
print("\n" + "="*50)
print("VERGLEICH: 30-MINUTEN vs 24-STUNDEN VORHERSAGE")
print("="*50)
print(f"30-Minuten Vorhersage - Test MAE: {mae_test_total:.4f} kW")
print(f"24-Stunden Vorhersage  - Test MAE: {mae_test_total_24h:.4f} kW")
# --- VERGLEICHSMETRIKEN FÜR 24H-VORHERSAGE ---

In [ ]:


# Berechne alle Metriken für 24h
rmse_test_24h = np.sqrt(mean_squared_error(y_test_multi, y_test_pred_24h))
mape_test_24h = mean_absolute_percentage_error(y_test_multi, y_test_pred_24h)
r2_test_24h = r2_score(y_test_multi, y_test_pred_24h)

# Metriken pro Schritt
rmse_per_step_24h = np.sqrt(mean_squared_error(y_test_multi, y_test_pred_24h, multioutput='raw_values'))
mape_per_step_24h = np.array([mean_absolute_percentage_error(y_test_multi[:, i], y_test_pred_24h[:, i]) for i in range(forecast_horizon)])

print("="*60)
print("AUSFÜHRLICHE METRIKEN - 24-STUNDEN VORHERSAGE")
print("="*60)

# Gesamtmetriken
metrics_24h = {
    'MAE': mae_test_total_24h,
    'RMSE': rmse_test_24h,
    'MAPE (%)': mape_test_24h,
    'R²': r2_test_24h
}

metrics_df_24h = pd.DataFrame.from_dict(metrics_24h, orient='index', columns=['Wert'])
metrics_df_24h.index.name = 'Metrik'
print("Gesamtmetriken:")
print(metrics_df_24h.round(4))

# Metriken pro Stunde
print("\nMetriken pro Vorhersagestunde:")
metrics_per_hour = pd.DataFrame({
    'Stunde': range(1, forecast_horizon + 1),
    'MAE': mae_test_per_step_24h,
    'RMSE': rmse_per_step_24h,
    'MAPE (%)': mape_per_step_24h
})
print(metrics_per_hour.round(4).head(10))  # Erste 10 Stunden

In [ ]:


# --- VERGLEICH 30MIN vs 24H ---
print("\n" + "="*60)
print("DIRECT COMPARISON: 30-MINUTEN vs 24-STUNDEN")
print("="*60)

comparison = pd.DataFrame({
    'Metrik': ['MAE (kW)', 'RMSE (kW)', 'MAPE (%)', 'R²'],
    '30_Minuten': [mae_test_total, rmse_test, mape_test, r2_test],
    '24_Stunden': [mae_test_total_24h, rmse_test_24h, mape_test_24h, r2_test_24h],
    'Differenz': [mae_test_total_24h - mae_test_total, 
                 rmse_test_24h - rmse_test, 
                 mape_test_24h - mape_test, 
                 r2_test_24h - r2_test]
})

print(comparison.round(4))

# --- VISUALISIERUNG DES VERGLEICHS ---
plt.figure(figsize=(14, 6))

# MAE-Vergleich
plt.subplot(1, 2, 1)
plt.bar(['30 Minuten', '24 Stunden'], [mae_test_total, mae_test_total_24h])
plt.title('Vergleich: MAE')
plt.ylabel('MAE (kW)')
plt.grid(True, alpha=0.3)

# R²-Vergleich
plt.subplot(1, 2, 2)
plt.bar(['30 Minuten', '24 Stunden'], [r2_test, r2_test_24h])
plt.title('Vergleich: R²')
plt.ylabel('R² Score')
plt.ylim(0, 1)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('comparison_30min_vs_24h.png', dpi=300, bbox_inches='tight')
plt.show()

# --- FEATURE IMPORTANCE FÜR 24H ---
feature_importance_24h = pd.DataFrame({
    'feature': all_features,
    'coefficient': ridge_model_24h.coef_[0],
    'abs_importance': np.abs(ridge_model_24h.coef_[0])
})
feature_importance_24h = feature_importance_24h.sort_values('abs_importance', ascending=False)

print("\nTop 10 wichtigste Features für 24h-Vorhersage:")
print(feature_importance_24h.head(10)[['feature', 'coefficient']].to_string(index=False))

In [ ]:

# Zusätzliche Analyse: Tagesgang der Vorhersagegenauigkeit
hourly_accuracy = pd.DataFrame({
    'Stunde': range(24),
    'MAE': [np.mean(mape_per_step_24h[i::24]) for i in range(24)]  # MAE pro Tagesstunde
})

plt.figure(figsize=(12, 5))
plt.plot(hourly_accuracy['Stunde'], hourly_accuracy['MAE'], 'o-')
plt.title('Vorhersagegenauigkeit über den Tagesverlauf (24h-Vorhersage)')
plt.xlabel('Stunde des Tages')
plt.ylabel('MAE (kW)')
plt.xticks(range(0, 24, 2))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot der Performance-Degradation
hours = range(1, 25)
mae_values = mae_test_per_step_24h

plt.figure(figsize=(10, 6))
plt.plot(hours, mae_values, 'o-', linewidth=2, markersize=6)
plt.axhline(y=mae_test_total, color='r', linestyle='--', label='30min MAE (0.33 kW)')
plt.title('Performance-Degradation mit zunehmendem Vorhersagehorizont\nLineares Modell vs. 30min Benchmark', fontsize=14)
plt.xlabel('Vorhersagehorizont (Stunden)')
plt.ylabel('MAE (kW)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(range(1, 25, 2))
plt.tight_layout()
plt.savefig('performance_degradation.png', dpi=300, bbox_inches='tight')
plt.show()

# Berechne die Degradation
degradation = (mae_values[-1] - mae_values[0]) / mae_values[0] * 100
print(f"Performance-Degradation über 24 Stunden: {degradation:.1f}%")
rmse_30min = 0.5649
rmse_24h = 0.5950
mae_30min = 0.3301  
mae_24h = 0.4545

In [ ]:



# Relative Unterschiede
rmse_increase = (rmse_24h - rmse_30min) / rmse_30min * 100  # +5.3%
mae_increase = (mae_24h - mae_30min) / mae_30min * 100      # +37.7%

# --- FEHLERVERGLEICH 30MIN vs 24H ---

# 1. Erneut die 30-Minuten Vorhersage laden oder neu berechnen
# (Falls nicht mehr im Speicher, kurz neu berechnen)

# 2. Alternativ: Nur 24h Fehler analysieren
errors_24h = (y_test_multi - y_test_pred_24h).flatten()

# 3. Detaillierte RMSE Analyse für 24h
print("="*60)
print("DETAILIERTE RMSE ANALYSE - 24-STUNDEN VORHERSAGE")
print("="*60)

# RMSE pro Stunde
rmse_per_hour = np.sqrt(mean_squared_error(y_test_multi, y_test_pred_24h, multioutput='raw_values'))

# RMSE über den Tagesverlauf
rmse_by_hour_of_day = []
for hour in range(24):
    # RMSE für jede Tagesstunde berechnen (z.B. alle 13:00 Uhr Vorhersagen)
    hour_mask = df_test_clean.index[-(len(y_test_multi)):].hour == hour
    if np.any(hour_mask):
        rmse_hour = np.sqrt(mean_squared_error(y_test_multi[hour_mask], y_test_pred_24h[hour_mask]))
        rmse_by_hour_of_day.append(rmse_hour)
    else:
        rmse_by_hour_of_day.append(np.nan)

# Visualisierung RMSE über Tagesverlauf
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(range(1, 25), rmse_per_hour, 'o-', linewidth=2)
plt.title('RMSE über Vorhersagehorizont (24h)')
plt.xlabel('Stunde in die Zukunft')
plt.ylabel('RMSE (kW)')
plt.grid(True, alpha=0.3)
plt.xticks(range(1, 25, 4))

plt.subplot(1, 2, 2)
plt.plot(range(24), rmse_by_hour_of_day, 'o-', linewidth=2)
plt.title('RMSE über Tageszeit (24h-Vorhersage)')
plt.xlabel('Uhrzeit')
plt.ylabel('RMSE (kW)')
plt.grid(True, alpha=0.3)
plt.xticks(range(0, 24, 4))

plt.tight_layout()
plt.show()

In [ ]:


# 4. Analyse der großen Fehler
large_errors_mask = np.abs(errors_24h) > 1.0  # Fehler > 1 kW
large_errors_count = np.sum(large_errors_mask)
large_errors_percentage = large_errors_count / len(errors_24h) * 100

print(f"Anzahl großer Fehler (>1 kW): {large_errors_count} ({large_errors_percentage:.1f}%)")
print(f"Maximaler Fehler: {np.max(np.abs(errors_24h)):.3f} kW")
print(f"RMSE / MAE Ratio: {rmse_test_24h / mae_test_total_24h:.3f}")
print("> 1.0 bedeutet: Einige große Fehler treiben den RMSE hoch")

# 5. Zeitpunkt der größten Fehler analysieren
max_error_idx = np.argmax(np.abs(errors_24h))
max_error_time = df_test_clean.index[-(len(y_test_multi)):][max_error_idx // 24]
max_error_hour = max_error_time.hour

print(f"Größter Fehler trat auf um: {max_error_time}")
print(f"Uhrzeit: {max_error_hour}:00 Uhr")



In [ ]:
# 6. Zusammenhang mit Verbrauchslevel
high_consumption_mask = y_test_multi.flatten() > 2.0  # Hoher Verbrauch > 2kW
rmse_high_consumption = np.sqrt(mean_squared_error(y_test_multi.flatten()[high_consumption_mask], 
                                                   y_test_pred_24h.flatten()[high_consumption_mask]))

low_consumption_mask = y_test_multi.flatten() < 0.5  # Niedriger Verbrauch < 0.5kW
rmse_low_consumption = np.sqrt(mean_squared_error(y_test_multi.flatten()[low_consumption_mask], 
                                                  y_test_pred_24h.flatten()[low_consumption_mask]))

print(f"RMSE bei hohem Verbrauch (>2 kW): {rmse_high_consumption:.3f} kW")
print(f"RMSE bei niedrigem Verbrauch (<0.5 kW): {rmse_low_consumption:.3f} kW")

In [ ]:
# Visualisierung der Problem-Zeiten
problem_hours = [18, 19, 20, 21, 22]  # Abend-Spitzen
problem_mask = df_test_clean.index[-(len(y_test_multi)):].hour.isin(problem_hours)

rmse_problem_hours = np.sqrt(mean_squared_error(y_test_multi[problem_mask], y_test_pred_24h[problem_mask]))
rmse_non_problem_hours = np.sqrt(mean_squared_error(y_test_multi[~problem_mask], y_test_pred_24h[~problem_mask]))

print(f"RMSE während Abend-Spitzen (18-22 Uhr): {rmse_problem_hours:.3f} kW")
print(f"RMSE außerhalb Abend-Spitzen: {rmse_non_problem_hours:.3f} kW")
print(f"Verschlechterung während Spitzenzeiten: {(rmse_problem_hours/rmse_non_problem_hours-1)*100:.1f}%")